# ISOM 835 · Session 11: Forecasting Bike Demand
**Suffolk University · Sawyer Business School · Fall 2026 · Mon Nov 30 · Prof. Hasan Arslan**

**Tonight in one sentence:** so far every model predicted something about a row we could see (this booking, this customer). Tonight we predict **the future of a number**: how many bikes people will rent **tomorrow, hour by hour**. Time changes three rules: how we split, what counts as a baseline, and which columns we are allowed to use.

**How to use this notebook:** run it top to bottom with **Shift + Enter**. It takes about one minute. Two cells marked **OPTIONAL** install extra tools; skip them if you are short on time. Whenever you see **Stop and notice**, read it before you run the next cell.

| Part | The question | The tool |
|---|---|---|
| 1 | What does demand look like over time? | one picture, three rhythms |
| 2 | Why does a random split lie here? | the same model, two scores |
| 3 | What is the number to beat? | naive baselines |
| 4 | Can a model beat it? | lag features + gradient boosting |
| 5 | Where does it still miss, and what do we do about it? | errors by hour, a safety buffer |
| 6 | Can a model forecast with no training at all? | a foundation model (OPTIONAL) |

In [ ]:
# Environment check — run this cell first. If it fails in Colab: run  !pip install -q -U scikit-learn pandas  then Runtime → Restart session.
import sys, re, sklearn, pandas as pd, numpy as np
need = {'scikit-learn': ('1.6', sklearn.__version__), 'pandas': ('2.2', pd.__version__), 'numpy': ('1.26', np.__version__)}
v = lambda s: tuple(int(x) for x in re.findall(r'\d+', s)[:2])
old = {k: have for k, (want, have) in need.items() if v(have) < v(want)}
assert not old, f'please upgrade {old}: !pip install -q -U ' + ' '.join(old)
print(f'Python {sys.version.split()[0]} ·', ' · '.join(f'{k} {have}' for k, (_, have) in need.items()), '✓')

## The business situation: tomorrow's morning rush

You run operations for **Capital Bikeshare** in Washington, DC. People pick up a bike at one station and drop it at another. Every night, your vans move bikes around the city so the stations are ready for the morning.

Your boss asks one question every evening:

> *"How many rentals will start each hour **tomorrow**? I need to know how many bikes to have ready for the 8 AM rush."*

- **Too few bikes:** commuters find empty docks, and they stop trusting the service.
- **Too many bikes:** vans and drivers worked all night for nothing.

The data is **real**: every hour of 2011 and 2012, about 17,000 hours, with the weather and the calendar. It comes from a well known research dataset (Fanaee-T and Gama, 2013).

**The rule of the game:** our forecast runs **at midnight** and covers **the next 24 hours**. So at forecast time we know everything up to midnight, and nothing after.

In [ ]:
import warnings
import matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import train_test_split, TimeSeriesSplit, cross_val_score
from sklearn.metrics import mean_absolute_error
warnings.filterwarnings('ignore')        # hide version notices from the libraries

bikes = pd.read_csv('https://raw.githubusercontent.com/harslan/isom-835/master/public/data/bike_hour.csv.gz')
print(f'{len(bikes):,} rows, one per hour, from {bikes.dteday.min()} to {bikes.dteday.max()}')
bikes.head(3)

## 1. Meet the data: one row is one hour

| column | means |
|---|---|
| `dteday`, `hr` | the date and the hour (0 = midnight to 1 AM, 8 = 8 to 9 AM) |
| `cnt` | **rentals that started in that hour. This is what we forecast.** |
| `workingday`, `holiday` | 1 if it was a working day, 1 if it was a public holiday |
| `weathersit` | 1 = clear, 2 = mist, 3 = light rain or snow, 4 = heavy rain |
| `temp`, `hum`, `windspeed` | temperature, humidity and wind, scaled to 0 to 1 |
| `casual`, `registered` | the two kinds of riders that add up to `cnt` |

The first row: midnight on Jan 1, 2011, a Saturday, clear sky, **16 rentals**.

Time series need one more thing before anything else: **a clock with no missing ticks**. The next cell builds a proper timestamp and checks for hours with no row.

In [ ]:
bikes['ts'] = pd.to_datetime(bikes['dteday']) + pd.to_timedelta(bikes['hr'], unit='h')
bikes = bikes.set_index('ts').asfreq('h')            # one row for EVERY hour; missing hours appear as empty rows

gaps = bikes['cnt'].isna()
print(f'hours with no data: {gaps.sum()} out of {len(bikes):,}')
print('the days with the most missing hours:')
print(bikes.index[gaps].normalize().value_counts().head(3).to_string())

bikes['cnt'] = bikes['cnt'].interpolate()            # fill each gap with a straight line between its neighbors
for col in ['workingday', 'holiday', 'weathersit']:
    bikes[col] = bikes[col].ffill()
for col in ['temp', 'hum', 'windspeed']:
    bikes[col] = bikes[col].interpolate()
y = bikes['cnt']

**Stop and notice:** 165 hours are missing, and **Oct 29 and 30, 2012** top the list. That is **Hurricane Sandy**. DC shut down its transit and the system closed. Real data has holes, and the holes often have a story. Here we fill them with a straight line so the clock has no gaps. In a real job you would ask: *should a closed system count as zero rentals?*

## 2. Look before you model: three rhythms

The next cell draws the same series at three zoom levels.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 3.4))
y.resample('W').mean().plot(ax=ax[0], color='#5b6bd8', title='Two years: weekly average per hour')
week = y['2012-09-10':'2012-09-16']
week.plot(ax=ax[1], color='#1aa38a', title='One week in Sept 2012 (Mon to Sun)')
for a in ax[:2]: a.set_xlabel('')
y.groupby([y.index.dayofweek >= 5, y.index.hour]).mean().unstack(0).plot(ax=ax[2], color=['#5b6bd8', '#f5a524'], title='Average day: weekday vs weekend')
ax[2].legend(['weekday', 'weekend']); ax[2].set_xlabel('hour of day')
for a in ax: a.set_ylabel('rentals per hour')
plt.tight_layout(); plt.show()

**How to read it:**
- **Left, the year:** demand climbs in spring, peaks in summer, falls in winter. And 2012 is busier than 2011: the system was growing.
- **Middle, the week:** on weekdays there are **two spikes**, around 8 AM and 5 to 6 PM. The weekend looks different.
- **Right, the day:** weekdays are the **commute**: two sharp peaks. Weekends are one gentle **leisure hill** in the afternoon.

**Stop and notice:** this series has a **daily** rhythm, a **weekly** rhythm and a **yearly** rhythm, plus growth. Any forecast has to respect all of them. Hold on to the two weekday spikes. The 8 AM spike is our boss's question.

## 3. The trap: a random split that sees the future

Our test set is the **last 8 weeks** of 2012 (Nov 6 to Dec 31). That is the honest setup: train on the past, test on what came after.

But first, a trap. Let's train the **same model** with the **same columns** twice:
1. with the usual **random** split (`train_test_split`, which we used all semester), and
2. with a **time** split: everything before Nov 6 for training, the last 8 weeks for testing.

The columns are the calendar and the weather: hour, weekday, month, year, holiday, temperature, humidity and so on.

🎮 **Play it first:** [Station 1 of the Forecasting Explorer](https://isom-835.vercel.app/explore/forecast/#s1) lets you drag the split line yourself.

In [ ]:
calendar = pd.DataFrame({'hour': y.index.hour, 'weekday': y.index.dayofweek, 'month': y.index.month,
                         'year': y.index.year - 2011, 'workingday': bikes['workingday'], 'holiday': bikes['holiday'],
                         'weathersit': bikes['weathersit'], 'temp': bikes['temp'], 'hum': bikes['hum'],
                         'wind': bikes['windspeed']}, index=y.index)
cut = pd.Timestamp('2012-11-06')                                   # the last 8 weeks are the future
model = lambda: HistGradientBoostingRegressor(learning_rate=0.05, max_iter=600, random_state=835)

X_a, X_b, y_a, y_b = train_test_split(calendar, y, test_size=0.2, random_state=835)
random_mae = mean_absolute_error(y_b, model().fit(X_a, y_a).predict(X_b))

past, future = calendar.index < cut, calendar.index >= cut
time_mae = mean_absolute_error(y[future], model().fit(calendar[past], y[past]).predict(calendar[future]))

print(f'random split:  average miss {random_mae:5.1f} rentals per hour')
print(f'time split:    average miss {time_mae:5.1f} rentals per hour')
print(f'the random split looks {time_mae / random_mae:.1f} times better than the truth')

**Stop and notice:** same model, same columns, and the random split says the model misses by about **25 rentals an hour**. The honest time split says about **48**. The random split looks **almost twice as good** as it really is.

**Why?** A random split puts Tuesday 8 AM in the test set and Tuesday 7 AM and 9 AM, and Monday and Wednesday 8 AM, in the training set. The model just fills the hole between its neighbors. That is **interpolation**, not forecasting. Tomorrow has no neighbors from tomorrow.

> **Rule 1 of forecasting:** split by time. Train on the past, test on the future. Never shuffle.

You have seen this before. In Session 9 we tested the hotel model on later bookings, and its score dropped. Same lesson, louder.

## 4. The number to beat: naive baselines

Before any model, ask: *what would a person with no model do?* Three simple guesses for each hour of tomorrow:

| baseline | the guess for 8 AM tomorrow |
|---|---|
| **yesterday** | rentals at 8 AM today |
| **same day last week** | rentals at 8 AM, 7 days ago |
| **last 7 days average** | the average hour over the past week |

All three only use what we know at midnight. Our score is **MAE**: the average miss in rentals per hour. (We skip percentage errors: at 4 AM there are often only 2 or 3 rentals, and a miss of 3 would look like a 100 percent error.)

🎮 **Play it first:** [Station 2: Beat yesterday](https://isom-835.vercel.app/explore/forecast/#s2)

In [ ]:
test = y[future]
baselines = {'yesterday':              y.shift(24)[future],
             'same day last week':     y.shift(168)[future],
             'last 7 days average':    y.shift(24).rolling(168).mean()[future]}
for name, guess in baselines.items():
    print(f'{name:22s} average miss {mean_absolute_error(test, guess):6.1f} rentals per hour')

**How to read it:** "yesterday" misses by about **64** rentals an hour, "same day last week" by about **73**, and the flat 7 day average by about **135**. The average throws away the daily rhythm, so it is the worst.

**Stop and notice:** usually "same day last week" wins on weekly data. Here **yesterday** wins. Why? The last 8 weeks of 2012 include **Thanksgiving and Christmas**, and it gets colder every week. Last week is old news. Yesterday knows about the cold.

> **The number to beat tonight: 64 rentals per hour.** If a model cannot beat "same hour yesterday", it is not a model.

## 5. OPTIONAL: a classical forecaster

Statisticians have forecast series like this for 60 years. **StatsForecast** (free, by Nixtla) fits classic models in seconds. **MSTL** splits the series into a daily rhythm, a weekly rhythm and a trend, then extends each one into tomorrow. We run it exactly like our baselines: every midnight, forecast the next 24 hours, for all 56 test days.

This cell installs a package and takes about a minute. **Skip it if you are short on time;** nothing later depends on it.

In [ ]:
# OPTIONAL — installs statsforecast (about 1 minute). Nothing later depends on this cell.
try:
    from statsforecast import StatsForecast
except ImportError:
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'statsforecast'], check=True)
    from statsforecast import StatsForecast
from statsforecast.models import MSTL, AutoETS

history = y[y.index >= cut - pd.Timedelta(weeks=12)]                # 12 weeks of history before each forecast is plenty
sf_input = pd.DataFrame({'unique_id': 'bikes', 'ds': history.index, 'y': history.values})
sf = StatsForecast(models=[MSTL(season_length=[24, 168], trend_forecaster=AutoETS(model='ZZN'))], freq='h')
sf_cv = sf.cross_validation(df=sf_input, h=24, step_size=24, n_windows=56)   # 56 midnights, 24 hours each
print(f"MSTL average miss: {mean_absolute_error(sf_cv['y'], sf_cv['MSTL']):.1f} rentals per hour")

**How to read it:** MSTL beats "yesterday" (about **59** against **64**), but not by much. Classical models only see the **past of the series itself**. They do not know that tomorrow is a holiday or that rain is coming. That is their limit, and it is our opening.

## 6. Turn forecasting into regression: lag features

Here is the big idea of tonight. We already know how to predict a number from columns: **gradient boosting**, Session 7. So we turn the past into columns.

For every hour, we add:

| new column | means | known at midnight? |
|---|---|---|
| `lag_24` | rentals in the same hour **yesterday** | ✅ yes |
| `lag_168` | rentals in the same hour **last week** | ✅ yes |
| `avg_last_week` | average rentals over the 7 days before yesterday ended | ✅ yes |
| `lag_1` | rentals **one hour ago** | ❌ **no**, for 8 AM tomorrow that is 7 AM tomorrow |

> **Rule 2 of forecasting:** every column must be **known at forecast time**. `lag_1` looks wonderful in a notebook and is impossible in real life. It is the time series version of the leaks we hunted all semester.

**One honest caveat.** The weather columns are the weather that **actually happened**. At midnight we would only have a **weather forecast**. Using the real weather is like having a perfect weather forecast, so our scores are a little optimistic. A real system would use the forecast instead.

🎮 **Play it first:** [Station 3: The 24 hour rule](https://isom-835.vercel.app/explore/forecast/#s3) lets you build lag columns and see which ones are allowed.

In [ ]:
features = calendar.copy()
features['lag_24'] = y.shift(24)                              # same hour yesterday
features['lag_168'] = y.shift(168)                            # same hour last week
features['avg_last_week'] = y.shift(24).rolling(168).mean()   # 7 day average, ending 24 hours ago
features = features.dropna()                                  # the first week has no "last week"
target = y.loc[features.index]

train = features.index < cut
X_train, y_train = features[train], target[train]
X_test, y_test = features[~train], target[~train]
print(f'training hours: {len(X_train):,}   test hours: {len(X_test):,}   columns: {features.shape[1]}')
features.loc['2012-11-12 08:00'].round(2)

**How to read it:** that is one row: **Nov 12, 2012 at 8 AM**, a Monday. It was Veterans Day, so `holiday` is 1. The model sees the hour, the day, the weather, and three numbers from the past: **119** rentals yesterday at 8 AM (a Sunday), **648** last Monday at 8 AM (a normal workday), and about **232** in last week's average hour. Which past number should it trust today? That is exactly what boosting learns.

## 7. Validate in time order, then train

To check the model **before** touching the test weeks, we use **`TimeSeriesSplit`**. It is cross validation with one rule: the validation block always comes **after** the training block. Five rounds, each one a "pretend future" of two weeks.

```
round 1:  [train ......]            [check]
round 2:  [train ..........]        [check]
round 3:  [train ..............]    [check]
```

In [ ]:
gbm = HistGradientBoostingRegressor(learning_rate=0.05, max_iter=2000, early_stopping=True, random_state=835)
folds = TimeSeriesSplit(n_splits=5, test_size=24 * 14)               # 5 rounds, each checks the next 2 weeks
cv_mae = -cross_val_score(gbm, X_train, y_train, cv=folds, scoring='neg_mean_absolute_error')
print('average miss in each pretend future:', cv_mae.round(1), f'  mean {cv_mae.mean():.1f}')

gbm.fit(X_train, y_train)
pred = pd.Series(gbm.predict(X_test), index=X_test.index)
best_baseline = mean_absolute_error(y_test, X_test['lag_24'])
model_mae = mean_absolute_error(y_test, pred)
print(f'\nTEST, last 8 weeks:  yesterday {best_baseline:.1f}   ->   model {model_mae:.1f} rentals per hour')
print(f'the model misses {1 - model_mae / best_baseline:.0%} less than the best baseline')

**Stop and notice:**
- In the five pretend futures the model misses by about **32 to 48** rentals an hour. The rounds differ because some two week blocks are calm and others have storms or holidays. That spread is a **range of honest answers**, not one number.
- On the real test weeks, the model misses by about **40**, against **64** for "yesterday". That is about **38 percent better**. A real win.

> **Rule 3 of forecasting:** a model only counts if it beats the naive baseline **on the future**.

## 8. Where does it still miss?

An average hides a lot. The next cell splits the misses by **hour of day**, and plots one test week: the truth, "yesterday" and the model.

In [ ]:
misses = pd.DataFrame({'model': (y_test - pred).abs(), 'yesterday': (y_test - X_test['lag_24']).abs()})
fig, ax = plt.subplots(1, 2, figsize=(15, 3.6))
misses.groupby(misses.index.hour).mean().plot(ax=ax[0], color=['#1aa38a', '#d64545'], marker='o', ms=3,
                                              title='Average miss by hour of day (test weeks)')
ax[0].set_xlabel('hour of day'); ax[0].set_ylabel('rentals per hour')
wk = slice('2012-11-19', '2012-11-25')                     # Thanksgiving week
ax[1].plot(y_test[wk].index, y_test[wk], color='black', lw=1.5, label='what happened')
ax[1].plot(y_test[wk].index, X_test['lag_24'][wk], color='#d64545', lw=1, ls='--', label='yesterday')
ax[1].plot(y_test[wk].index, pred[wk], color='#1aa38a', lw=1.2, label='model')
ax[1].set_title('Thanksgiving week 2012 (Mon to Sun)'); ax[1].legend()
plt.tight_layout(); plt.show()

worst = misses.groupby(misses.index.date).mean().sort_values('model', ascending=False).head(4).round(0)
print('the 4 days the model missed most (average miss per hour):')
print(worst.to_string())

**How to read it:**
- **Left:** both lines are low at night and high at the commute peaks. The model's worst hour is **8 AM**: it misses by about 90 rentals there. "Yesterday" misses by about 180, twice as much. The model helps most exactly where the boss cares most.
- **Right:** on Thursday, **Thanksgiving**, the commute disappears (black line). "Yesterday" (red dashes) still expects the Wednesday rush: 551 rentals at 8 AM. The model (green) knows it is a holiday and cuts 8 AM to about 300, but still too many: only 97 came. And it still expects an evening rush that never happens.
- **The worst days:** Thanksgiving, Christmas Day, the day after Thanksgiving and Christmas Eve. **Holidays are the hard part** of every demand forecast.
- **Look at Christmas:** there the model did **worse** than "yesterday". Christmas Eve is not a holiday in this data, so the model expected a normal working morning: it forecast about 470 rentals at 8 AM, and 66 came. Then on Christmas Day, "yesterday" was already quiet, so the lazy guess won. A better holiday calendar would fix both. That is why companies like DoorDash build special holiday models (see the resources on the session page).

## 9. From a forecast to a decision: the safety buffer

The boss does not want an average miss. She wants to know **how many bikes to have ready at 8 AM**.

If we stage exactly the forecast, we will run short on many mornings, because the forecast is sometimes too low. So we add a **buffer**: forecast **plus** a few extra bikes. A bigger buffer means fewer empty docks, but more bikes moved for nothing. Operations people call it **safety stock**.

The next cell checks the **37 working day mornings** in the test weeks: how often would we have run short with each buffer?

In [ ]:
mornings = (y_test.index.hour == 8) & (X_test['workingday'] == 1)
need, forecast = y_test[mornings], pred[mornings]
print(f'{mornings.sum()} working day mornings at 8 AM, average demand {need.mean():.0f} rentals\n')
for extra in [0, 50, 100, 150]:
    short = (need > forecast + extra).sum()
    print(f'stage the forecast + {extra:3d} bikes  ->  short on {short:2d} of {mornings.sum()} mornings')

**Stop and notice:** with **no buffer** we run short on **28 of 37** mornings. With **150 extra bikes** we run short on only **4**. Now the boss has a real choice, in her own language: *"How many short mornings can we live with, and what do the extra bikes cost?"*

> A forecast is not the decision. **The forecast plus a buffer, chosen by the cost of being short, is the decision.** (Session 8 did the same thing with a threshold.)

🎮 **Play it first:** [Station 4: How big a buffer?](https://isom-835.vercel.app/explore/forecast/#s4)

## 10. OPTIONAL: a foundation model that has never seen this data

In 2025, foundation models for time series became really good. **Chronos-2** (Amazon, Apache-2.0 license) was trained on a huge collection of real and synthetic series. You hand it the history, and it forecasts with **no training at all**. It can also use columns like temperature if you give them.

We run it the same way: at each of the 56 midnights, give it 8 weeks of history and ask for the next 24 hours. It downloads about 500 MB the first time. On a Colab **GPU** runtime it takes under a minute; on CPU, a few minutes. Nothing later depends on this cell.

In [ ]:
# OPTIONAL — installs chronos-forecasting and downloads the Chronos-2 model. Nothing later depends on this cell.
try:
    from chronos import Chronos2Pipeline
except ImportError:
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'chronos-forecasting', 'pandas[pyarrow]'], check=True)
    from chronos import Chronos2Pipeline
import torch
pipeline = Chronos2Pipeline.from_pretrained('amazon/chronos-2', device_map='cuda' if torch.cuda.is_available() else 'cpu')

cov = ['temp', 'hum', 'workingday', 'holiday']
histories, horizons = [], []
for day in pd.date_range(cut, periods=56, freq='D'):                  # each midnight in the test weeks
    past_hours = bikes.loc[day - pd.Timedelta(weeks=8): day - pd.Timedelta(hours=1)]
    histories.append(past_hours[cov].assign(id=str(day.date()), timestamp=past_hours.index, target=past_hours['cnt']))
    next_day = bikes.loc[day: day + pd.Timedelta(hours=23)]
    horizons.append(next_day[cov].assign(id=str(day.date()), timestamp=next_day.index))
chronos = pipeline.predict_df(pd.concat(histories), future_df=pd.concat(horizons), prediction_length=24,
                              quantile_levels=[0.1, 0.5, 0.9], id_column='id', timestamp_column='timestamp', target='target')
chronos = chronos.set_index('timestamp')['predictions']
common = chronos.index.intersection(y_test.index)
print(f'Chronos-2, zero training:  average miss {mean_absolute_error(y_test[common], chronos[common]):.1f} rentals per hour')
print(f'our boosted model:         average miss {mean_absolute_error(y_test[common], pred[common]):.1f}')

**How to read it:** compare the two numbers. A model that has **never seen** Washington's bikes, with no training at all, against the model we built tonight. Whoever wins, the lesson is the same:

> A foundation model is the **new first baseline**. It costs three lines. If your carefully built model cannot beat it, ship the foundation model.

## 11. Your turn

Work in the empty cell below. Pick any two.

1. **One more lag.** Add `lag_48` (same hour two days ago) to `features`, retrain, and print the test MAE. Does it help?
2. **Is lag_1 really that tempting?** Add `y.shift(1)` as a column and retrain. Watch the MAE drop, then write one sentence on why you could **never** use this model at midnight.
3. **Holidays.** Print the model's average miss on holiday hours and on normal hours (`X_test['holiday'] == 1`). How much worse are holidays?
4. **The buffer.** Your boss says: *"Short on at most 1 morning in 10."* Find the smallest buffer that does it.
5. **Your project.** Does your data have a clock? If yes, write the one line that splits it by time, and name your naive baseline.

In [ ]:
# Your turn — work here

## What we learned tonight

- **Split by time.** A random split let the model peek at the future and looked almost **twice as good** as the truth.
- **Beat the naive baseline first.** "Same hour yesterday" missed by about 64 rentals an hour. Every model is judged against that.
- **Forecasting is regression on the past.** Lag columns turn yesterday and last week into features that gradient boosting understands. Our model missed by about 40.
- **Every column must be known at forecast time.** `lag_1` is a leak in disguise.
- **Look at where it misses.** The 8 AM rush and the holidays are the hard part.
- **A forecast is not a decision.** Forecast plus a buffer, chosen by the cost of running short, is.
- **Foundation models** like Chronos-2 forecast with no training. They are the new first baseline.

**For your project:** if your data has a clock (dates, seasons, quarters), use a time split in Milestone 3, and report your model next to a naive baseline. Questions? Office hours or email.